# MuscleMap WB Segmentation — Sheffield Dataset (Lambda)

Runs **MuscleMap whole-body** segmentation on the 69 Sheffield augmented DICOM volumes.
Each DICOM is converted to NIfTI and passed to `mm_segment -r wholebody`.

MuscleMap requires Python 3.11; a conda env is created automatically.

Data: `~/sheffeld/20440164/Aug_N.dcm`
Output: `~/musclemap_wb_sheffield_segs/Aug_N_dseg.nii.gz`

⚠️ **Run this notebook before** `lambda_medclipsamv2_textboxes_sheffield.ipynb` and
`lambda_slmsam_sheffield.ipynb` — they use the WB segmentations as bounding-box prompts.

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os, shutil, glob, re
import numpy as np

try:
    import pydicom
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pydicom', 'SimpleITK'])
    import pydicom
import SimpleITK as sitk

# ── conda env with Python 3.11 + MuscleMap ────────────────────────────────────
_conda_candidates = [
    shutil.which('conda'),
    os.path.expanduser('~/miniconda3/bin/conda'),
    os.path.expanduser('~/anaconda3/bin/conda'),
    '/opt/conda/bin/conda',
]
CONDA = next((p for p in _conda_candidates if p and os.path.exists(p)), None)
if CONDA is None:
    subprocess.check_call(['bash', '-c',
        'wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh '
        '-O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -p ~/miniconda3'])
    CONDA = os.path.expanduser('~/miniconda3/bin/conda')

ENV_NAME = 'musclemap_env'
ENV_DIR  = os.path.join(os.path.dirname(os.path.dirname(CONDA)), 'envs', ENV_NAME)
ENV_PY   = os.path.join(ENV_DIR, 'bin', 'python')
MM_BIN   = os.path.join(ENV_DIR, 'bin', 'mm_segment')

if not os.path.exists(ENV_PY):
    subprocess.check_call([CONDA, 'create', '-n', ENV_NAME, 'python=3.11', 'pip',
                           '-c', 'conda-forge', '--override-channels', '-y', '-q'])
subprocess.check_call([ENV_PY, '-m', 'pip', 'install', '-q',
                       'git+https://github.com/MuscleMap/MuscleMap.git'])
print('mm_segment exists:', os.path.exists(MM_BIN))

In [ ]:
IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
NII_DIR    = os.path.expanduser('~/sheffeld_nii')          # temporary NIfTI cache
OUTPUT_DIR = os.path.expanduser('~/musclemap_wb_sheffield_segs')

os.makedirs(NII_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm')) if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)
print(f'Found {len(dcm_files)} DICOM volumes')

def dicom_to_nifti(dcm_path, nii_path):
    if os.path.exists(nii_path):
        return
    ds  = pydicom.dcmread(dcm_path)
    arr = ds.pixel_array.astype(np.float32)
    ps  = getattr(ds, 'PixelSpacing', [1.0, 1.0])
    st  = float(getattr(ds, 'SliceThickness', 1.0))
    img = sitk.GetImageFromArray(arr)
    img.SetSpacing([float(ps[1]), float(ps[0]), st])
    sitk.WriteImage(sitk.Cast(img, sitk.sitkFloat32), nii_path)

run_env = os.environ.copy()
run_env.pop('MPLBACKEND', None)

for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    nii_path = os.path.join(NII_DIR, f'Aug_{idx}.nii.gz')
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_dseg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\nProcessing: Aug_{idx}')
    dicom_to_nifti(dcm_path, nii_path)

    subprocess.check_call([
        MM_BIN, '-i', nii_path, '-r', 'wholebody', '-o', OUTPUT_DIR, '-g', 'Y',
    ], env=run_env)
    # mm_segment names output after input stem; rename to our convention
    mm_out = os.path.join(OUTPUT_DIR, f'Aug_{idx}_dseg.nii.gz')
    if not os.path.exists(mm_out):
        candidates = glob.glob(os.path.join(OUTPUT_DIR, f'Aug_{idx}*dseg*.nii.gz'))
        if candidates:
            shutil.move(candidates[0], mm_out)
    print(f'  Saved → {mm_out}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(results[0]))
    print(f'Sample shape: {arr.shape}  labels: {sorted(np.unique(arr[arr>0]).tolist())[:10]}')